# Notebook 05 — Mecanismo Biológico: BUB1B/BUBR1 y el SAC

**Track 2 — MVA Hackathon 2026**

**Objetivo:** Describir con rigor científico qué hace la variante causante (BUB1B het. compuesta) y cuál es la consecuencia celular hasta llegar al fenotipo MVA.  
**Hitos:** H2 — Resumen a R4 (12 oct) · H3 — Mecanismo completo (14 oct)

---

## Mapa del notebook

```
1. Estructura y función de BUBR1 (BUB1B)
2. El Spindle Assembly Checkpoint (SAC) — mecanismo normal
3. Efecto molecular de p.Leu737Ter + p.Asn1002Lys
4. Consecuencia celular: SAC disfuncional → missegregación → mosaicismo
5. Conexión con el fenotipo clínico del paciente
6. Resumen para R4 (base para reposicionamiento)
7. Literatura clave
```

In [ ]:
# ── Celda 0: Dependencias ──────────────────────────────────────────────────
import json
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Notebook 05 — Mecanismo biológico BUB1B/BUBR1')
print('Dependencias cargadas.')

---
## 1. Estructura y función de BUBR1 (proteína codificada por BUB1B)

### 1.1 El gen BUB1B

| Campo | Detalle |
|-------|--------|
| **Gen** | BUB1B (BUB1 Mitotic Checkpoint Serine/Threonine Kinase B) |
| **Alias** | BUBR1, MAD3L, SSK1 |
| **Locus** | 15q15.1 (GRCh38: 15:40,160,984–40,221,137) |
| **Transcrito canónico** | ENST00000287598 (BUB1B-201), 23 exones, 1050 aa (Ensembl, verificado) |
| **Herencia MVA** | Autosómica recesiva (AR) |
| **Frecuencia en MVA** | ~50% de los casos son BUB1B bialélico |

### 1.2 Dominios proteicos de BUBR1 (1050 aminoácidos)

```
  1        200        400        600        737       1002     1050
  |---------|---------|---------|---------|  *  |------  *  |---|
  [KEN box] [TPR repeats x8 ]  [GLEBS]  [BUBR1-C] [kinase-like]
                                                    
  * p.Leu737Ter  → trunca en el dominio GLEBS/BUBR1-C
  * p.Asn1002Lys → en el pseudo-dominio kinasa
```

| Dominio | Residuos | Función |
|---------|----------|---------|
| KEN box | 1–60 | Reconocimiento por APC/C-Cdh1 (degradación) |
| TPR (x8) | 60–400 | Interacción con BUB3, reclutamiento al cinetocoro |
| GLEBS | 430–480 | Unión a BUB3 (esencial para localización cinetocoro) |
| BUBR1-C / linker | 550–780 | Unión a CDC20, formación del MCC |
| Pseudo-kinasa | 780–1050 | Regulación alostérica, estabilización MCC con p31^comet |

> ⚠ **Límites de dominios sin verificar (tarea B1).** Solo estos están confirmados en UniProt O60566: dominio *BUB1 N-terminal* 62–226 · región de interacción con KNL1 152–185 · caja D 224–232 · dominio *proteína quinasa* 766–1050. Los demás límites de la tabla (KEN, TPR, GLEBS, BUBR1-C) y los rótulos de la figura 02a vienen de memoria y deben contrastarse con la literatura antes del informe. En particular, p.Leu737 queda **antes** del dominio quinasa (766), no dentro de él.

**Función central:** BUBR1 es el componente catalítico del **Mitotic Checkpoint Complex (MCC)**:  
`MCC = BUBR1 : BUB3 : MAD2 : CDC20`  
El MCC inhibe APC/C-CDC20, bloqueando la transición metafase→anafase hasta que todos los cinetocoros estén correctamente anclados a los microtúbulos del huso mitótico.

In [ ]:
# ── Celda 1: Diagrama de dominios proteicos BUBR1 ─────────────────────────

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.set_xlim(0, 1100)
ax.set_ylim(-1.5, 3)
ax.axis('off')
ax.set_title('Dominios proteicos de BUBR1 (BUB1B, 1050 aa) — variantes en contexto estructural',
             fontsize=11, fontweight='bold', pad=10)

# Barra principal de la proteína
ax.add_patch(FancyBboxPatch((10, 0.5), 1040, 0.8,
             boxstyle='round,pad=0.02', facecolor='#BBDEFB', edgecolor='#1976D2', lw=1.5))

domains = [
    (1,   60,  '#FF8A65', 'KEN\nbox'),
    (60,  400, '#FFB74D', 'TPR repeats (x8)\n(unión BUB3)'),
    (430, 480, '#81C784', 'GLEBS\n(BUB3)'),
    (550, 780, '#64B5F6', 'BUBR1-C\n(CDC20/MCC)'),
    (780, 1050,'#CE93D8', 'Pseudo-kinasa\n(alostérico)'),
]

for start, end, color, label in domains:
    ax.add_patch(FancyBboxPatch((start, 0.52), end - start, 0.76,
                 boxstyle='round,pad=0.01', facecolor=color, edgecolor='white', lw=1, alpha=0.9))
    mid = (start + end) / 2
    ax.text(mid, 0.9, label, ha='center', va='center', fontsize=7.5, fontweight='bold')

# Variantes
variants = [
    (737,  'p.Leu737Ter\n(stop → NMD)', '#C00000', '▼'),
    (1002, 'p.Asn1002Lys\n(missense)', '#2E7D32', '▼'),
]
for pos, label, color, symbol in variants:
    ax.annotate('', xy=(pos, 1.28), xytext=(pos, 1.9),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.text(pos, 2.05, label, ha='center', va='bottom', fontsize=8,
            color=color, fontweight='bold')
    ax.axvline(x=pos, color=color, linestyle='--', lw=1.2, alpha=0.5, ymin=0.37, ymax=0.72)

# Escala en aa
for tick in [1, 200, 400, 600, 800, 1000]:
    ax.text(tick, 0.3, str(tick), ha='center', va='top', fontsize=7, color='gray')
    ax.plot([tick, tick], [0.5, 0.52], color='gray', lw=0.8)

ax.text(525, 0.25, 'Posición (aa)', ha='center', va='top', fontsize=8, color='gray')

# Región truncada
ax.add_patch(FancyBboxPatch((737, 0.52), 313, 0.76,
             boxstyle='round,pad=0.01', facecolor='none',
             edgecolor='#C00000', lw=2, linestyle='--', alpha=0.7))
ax.text(893, -0.2, '← 313 aa perdidos por p.Leu737Ter (NMD probable)', 
        ha='center', va='top', fontsize=7.5, color='#C00000', style='italic')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figura_02a_dominios_bubr1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: results/figura_02a_dominios_bubr1.png')

---
## 2. El Spindle Assembly Checkpoint (SAC) — mecanismo normal

### 2.1 ¿Qué es el SAC?

El SAC (también llamado **Mitotic Checkpoint**) es el sistema de vigilancia que garantiza que la célula no entre en anafase hasta que **cada cinetocoro esté correctamente unido a microtúbulos de polos opuestos** (unión bivalente correcta = bi-orientación).

### 2.2 Actores principales

| Proteína | Gen | Función en el SAC |
|----------|-----|-------------------|
| **BUBR1** | *BUB1B* | Componente catalítico del MCC; inhibe APC/C-CDC20 |
| BUB1 | *BUB1* | Kinasa; recluta BUB3 y BUBR1 al cinetocoro |
| BUB3 | *BUB3* | Adaptador; une BUBR1 a MAD2 |
| MAD1 | *MAD1L1* | Plataforma catalítica para activar MAD2 |
| MAD2 | *MAD2L1* | Inhibidor directo de CDC20 ("open"→"closed" MAD2) |
| CDC20 | *CDC20* | Activador de APC/C; secuestrado por MCC |
| APC/C | complejo | Ubiquitin ligasa; degrada securina y ciclina B |

### 2.3 Ciclo normal del SAC (paso a paso)

```
PROFASE/PROMETAFASE — cinetocoro sin microtúbulo:
  ┌─────────────────────────────────────────────────┐
  │ MAD1-MAD2(open) → MAD2(closed) activo           │
  │ MAD2(closed) + BUBR1:BUB3 + CDC20 → MCC         │ ← SAC ACTIVO
  │ MCC inhibe APC/C → securina y ciclina B estables │
  │ Separasas bloqueadas → cohesinas intactas         │
  └─────────────────────────────────────────────────┘
        ↓  cuando todos los cinetocoros se unen correctamente
METAFASE→ANAFASE — todos cinetocoros bi-orientados:
  ┌──────────────────────────────────────────────┐
  │ p31^comet deshace el MCC                     │
  │ CDC20 libre activa APC/C                     │ ← SAC APAGADO
  │ APC/C ubiquitina securina → separasa activa   │
  │ Separasa corta cohesinas → cromátidas separan │
  └──────────────────────────────────────────────┘
```

In [ ]:
# ── Celda 2: Figura — Vía SAC normal vs disfuncional ─────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
fig.patch.set_facecolor('#F8F9FA')

def draw_box(ax, x, y, w, h, text, facecolor, edgecolor='#333', fontsize=9, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle='round,pad=0.08', facecolor=facecolor,
                 edgecolor=edgecolor, lw=1.5))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold' if bold else 'normal',
            multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2, color='#555', style='->', lw=1.5):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=lw))

def draw_inhibit(ax, x1, y1, x2, y2, color='#C00000'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='-[', color=color, lw=2))

for ax, title, functional in [(ax1, 'SAC FUNCIONAL (normal)', True), 
                               (ax2, 'SAC DISFUNCIONAL (BUB1B het. compuesta)', False)]:
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 11)
    ax.axis('off')
    color_title = '#1565C0' if functional else '#C00000'
    ax.set_title(title, fontsize=11, fontweight='bold', color=color_title, pad=8)

    # Cinetocoro sin unir
    draw_box(ax, 3.5, 9.2, 3, 0.7, 'Cinetocoro sin microtúbulo', '#E3F2FD', '#1976D2')
    
    # MAD1-MAD2
    draw_box(ax, 3.5, 8.0, 3, 0.7, 'MAD1–MAD2(open)\n→ MAD2(closed)', '#E8F5E9', '#2E7D32', fontsize=8)
    draw_arrow(ax, 5, 9.2, 5, 8.7)

    if functional:
        # MCC formado
        draw_box(ax, 2.5, 6.6, 5, 0.9, 'MCC = BUBR1:BUB3:MAD2:CDC20\n(activo, intacto)', '#FFF9C4', '#F9A825', fontsize=8.5, bold=True)
        draw_arrow(ax, 5, 8.0, 5, 7.5)
        
        # APC/C inhibido
        draw_box(ax, 3.2, 5.2, 3.6, 0.7, 'APC/C-CDC20 INHIBIDO', '#FFECB3', '#FF8F00', fontsize=8)
        draw_inhibit(ax, 5, 6.6, 5, 5.9)
        ax.text(5.3, 6.25, '⊣', ha='center', va='center', fontsize=16, color='#C00000')
        
        # Securina / ciclina B estables
        draw_box(ax, 2.5, 3.8, 5, 0.7, 'Securina + Ciclina B estables\n(separasa bloqueada)', '#F3E5F5', '#7B1FA2', fontsize=8)
        draw_arrow(ax, 5, 5.2, 5, 4.5)
        
        # Cohesinas
        draw_box(ax, 3.0, 2.5, 4, 0.7, 'Cohesinas intactas', '#E8F5E9', '#388E3C', fontsize=8)
        draw_arrow(ax, 5, 3.8, 5, 3.2)
        
        # Resultado
        draw_box(ax, 2.5, 1.2, 5, 0.9, '✅ METAFASE CORRECTA\nSeparación ordenada de cromátidas', '#E8F5E9', '#1B5E20', fontsize=9, bold=True)
        draw_arrow(ax, 5, 2.5, 5, 2.1)
        
    else:
        # BUBR1 insuficiente
        draw_box(ax, 2.2, 6.6, 5.6, 0.9, 'BUBR1 insuficiente\n(1 alelo NMD + 1 alelo missense)', '#FFEBEE', '#C00000', fontsize=8, bold=True)
        draw_arrow(ax, 5, 8.0, 5, 7.5)
        ax.text(5, 7.25, '↓ cantidad y función', ha='center', fontsize=7.5, color='#C00000', style='italic')
        
        # MCC débil
        draw_box(ax, 2.5, 5.2, 5, 0.9, 'MCC débil / inestable\n(inhibición APC/C incompleta)', '#FFF3E0', '#E65100', fontsize=8)
        draw_arrow(ax, 5, 6.6, 5, 6.1)
        
        # APC/C activo prematuro
        draw_box(ax, 2.8, 3.9, 4.4, 0.8, 'APC/C-CDC20 ACTIVO PREMATURO', '#FFCDD2', '#B71C1C', fontsize=8, bold=True)
        draw_arrow(ax, 5, 5.2, 5, 4.7, color='#C00000')
        ax.text(5.4, 4.85, '✗', ha='center', va='center', fontsize=14, color='#C00000')
        
        # Separación prematura
        draw_box(ax, 2.2, 2.7, 5.6, 0.8, 'Securina degradada → separasa activa\nCohesinas cortadas PREMATURAMENTE', '#FFCDD2', '#C00000', fontsize=7.5)
        draw_arrow(ax, 5, 3.9, 5, 3.5, color='#C00000')
        
        # Resultado
        draw_box(ax, 1.5, 1.2, 7, 1.1,
                 '❌ MISSEGREGACIÓN CROMOSÓMICA\nCélulas hijas con +/- cromosomas (aneuploidía)\n→ Mosaicismo → MVA',
                 '#FFEBEE', '#B71C1C', fontsize=8.5, bold=True)
        draw_arrow(ax, 5, 2.7, 5, 2.3, color='#C00000')

plt.suptitle('Spindle Assembly Checkpoint (SAC): normal vs BUB1B insuficiente',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figura_02b_sac_normal_vs_bub1b.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: results/figura_02b_sac_normal_vs_bub1b.png')

---
## 3. Efecto molecular de las dos variantes

### 3.1 Alelo 1 — p.Leu737Ter (stop_gained)

| Aspecto | Detalle |
|---------|--------|
| **Posición** | 15:40209701 (GRCh38) · exón 17 de 23 (ENST00000287598) |
| **Cambio** | c.2210T>G → p.Leu737Ter (TTA→TGA) |
| **Consecuencia ARN** | Codón de parada prematuro (PTC) en el codón 737 de 1050. Si escapara al NMD, el producto tendría 736 aa (se pierden 314) |
| **Destino del ARNm** | **NMD predicho (tarea B2).** El PTC está en c.2210 (exón 17). La última unión exón-exón (exones 22/23) queda en c.2957/2958, **747 nt** después del PTC; la regla clásica pide > 50–55 nt. No está en el primer exón ni cerca del ATG. Es una predicción por regla: no hay datos de ARN del paciente que la confirmen |
| **Consecuencia proteína** | Proteína truncada o ausente (pérdida de función) |
| **Dominios perdidos** | Todo el dominio quinasa (766–1050, UniProt) y la región que lo precede (737–765). Se conservan el dominio BUB1 N-terminal (62–226) y la región de unión a KNL1 |
| **Criterio ACMG** | **PVS1** (muy fuerte) + PM2_Supporting. Detalle y puntaje en `docs/secciones/01_variante_causante.md` |

**Por qué el NMD importa:** Si el ARNm se destruye, el alelo no produce ninguna proteína BUBR1 funcional. La célula depende enteramente del alelo 2 (p.Asn1002Lys), que además es defectuoso.

### 3.2 Alelo 2 — p.Asn1002Lys (missense)

| Aspecto | Detalle |
|---------|--------|
| **Posición** | 15:40220612 (GRCh38) · dominio pseudo-kinasa |
| **Cambio** | c.3006T>G → p.Asn1002Lys |
| **Residuo** | N1002: asparagina → lisina (cambio de carga: neutro → positivo) |
| **Función del dominio** | El dominio quinasa de BUBR1 (766–1050) es una pseudoquinasa. Su papel funcional exacto (p. ej. interacción con p31^comet, estabilidad del MCC) está por verificar en la literatura (tarea B1) |
| **Impacto predicho** | Cambio de carga (neutro → positivo). La conservación del residuo y su entorno estructural están por evaluar (tarea B3) |
| **Predictores** | CADD v1.7 = 24,5 · AlphaMissense 0,92 (probablemente patogénica) · PolyPhen-2 1,0 y SIFT 0,0 (dañino) · **REVEL 0,472 (indeterminado)** · BayesDel (con AF) = 0,049 (tolerado) |
| **Criterio ACMG** | **PM2_Supporting** (1 alelo en gnomAD v4). PP3 discutible: los predictores discrepan. Formalmente **VUS**; ver `docs/secciones/01_variante_causante.md` |

### 3.3 Por qué la combinación es patogénica (modelo haploinsufiencia compuesta)

```
Célula normal:  2 alelos funcionales → BUBR1 suficiente → SAC robusto

Portador (1 alelo):  1 funcional + 1 mutado → ~50% BUBR1 → SAC competen-
                     te (heterocigótos sanos; abortos recurrentes leves)

Paciente (2 alelos): alelo NMD (= 0 proteína) + alelo missense (= proteína
                     disfuncional) → BUBR1 «casi cero» funcional → SAC roto
```

**Nota sobre los abortos recurrentes parentales (hipótesis, sin datos de los padres):** el dataset no incluye a los progenitores, así que no se sabe cuál alelo aporta cada uno. Si cada progenitor fuera portador de un alelo, tendría ~50 % de BUBR1 funcional, y se podría proponer inestabilidad cromosómica leve como contribuyente a los abortos. Esto es una conjetura, no un hallazgo.

---
## 4. Consecuencia celular: del SAC roto al mosaicismo

### 4.1 Cascada fisiopatológica

```
BUB1B het. compuesta
      ↓
BUBR1 insuficiente (cantidad ↓ + función ↓)
      ↓
MCC no inhibe APC/C eficientemente
      ↓
APC/C degrada securina y ciclina B ANTES de que todos los
cinetocoros estén bi-orientados
      ↓
Separasa activa → cohesinas cortadas prematuramente
      ↓
MISSEGREGACIÓN: cromátidas van al polo equivocado
      ↓
Células hijas con número anormal de cromosomas (aneuploidía)
      ↓  (ocurre en algunas células, no todas → mosaico)
MOSAICISMO ANEUPLO → MOSAIC VARIEGATED ANEUPLOIDY (MVA)
```

### 4.2 ¿Por qué mosaico y no uniforme?

- BUBR1 está **reducido pero no ausente** — algunas mitosis funcionan correctamente
- Las células con monosomías letales se eliminan → las aneuploidías viables persisten
- Resultado: tejidos con mezcla de células euploides y aneuploides

### 4.3 ¿Por qué rabdomiosarcoma y no otro tumor?

- La inestabilidad cromosómica (CIN) es un mecanismo pro-oncogénico general
- Las pérdidas de tumor supresores (p53, RB1) en células musculares fuerzan la expansión clonal
- BUB1B bialélico se asocia específicamente con rabdomiosarcoma y tumores de Wilms en MVA
- Referencia: Hanks et al. 2004 (Nat Genet); Rivero et al. 2016 (Eur J Hum Genet)

In [ ]:
# ── Celda 3: Figura — Cascada BUB1B → MVA → Fenotipo ─────────────────────

fig, ax = plt.subplots(figsize=(10, 12))
ax.set_xlim(0, 10)
ax.set_ylim(-0.5, 13)
ax.axis('off')
ax.set_title('Cascada fisiopatológica: BUB1B het. compuesta → MVA → Fenotipo clínico',
             fontsize=11, fontweight='bold', pad=10)

steps = [
    (12.2, 0.7, '#E3F2FD', '#1976D2', 'BUB1B het. compuesta\np.Leu737Ter (NMD) + p.Asn1002Lys (missense)', True),
    (11.2, 0.7, '#FFF3E0', '#E65100', 'BUBR1 insuficiente\n(cantidad ↓ + función ↓)', False),
    (10.1, 0.7, '#FFF3E0', '#E65100', 'MCC inestable\n(inhibición APC/C incompleta)', False),
    (9.0,  0.7, '#FFEBEE', '#C00000', 'APC/C activo prematuro\n(antes de bi-orientación completa)', False),
    (7.9,  0.7, '#FFEBEE', '#C00000', 'Cohesinas cortadas prematuramente\n(separasa activa)', False),
    (6.8,  0.7, '#FFEBEE', '#C00000', 'Missegregación cromosómica\n(cromátidas al polo equivocado)', False),
    (5.7,  0.7, '#FCE4EC', '#880E4F', 'Aneuploidía celular\n(ganancia / pérdida de cromosomas)', False),
    (4.6,  0.7, '#FCE4EC', '#880E4F', 'Mosaicismo variegado\n(mezcla de clones euploides y aneuploides)', True),
    (3.2,  0.7, '#EDE7F6', '#4527A0', 'Inestabilidad cromosómica crónica (CIN)\n→ Driver oncogénico (pérdida de supresores tumorales)', False),
    (1.8,  0.7, '#FFECB3', '#E65100', 'Prematuridad · RCIU · Talla baja\nNefrocalcinosis · Atrofia muscular', False),
    (0.4,  0.9, '#FFCDD2', '#B71C1C', '⚠ Rabdomiosarcoma\n(proliferación clonal de células aneuploides)', True),
]

y_positions = [12.0, 10.8, 9.6, 8.4, 7.2, 6.0, 4.8, 3.5, 2.3, 1.0, -0.2]
x0, w = 1.5, 7

for (_, h, fc, ec, text, bold), y in zip(steps, y_positions):
    ax.add_patch(FancyBboxPatch((x0, y), w, h,
                 boxstyle='round,pad=0.1', facecolor=fc, edgecolor=ec, lw=2 if bold else 1.2))
    ax.text(x0 + w/2, y + h/2, text, ha='center', va='center',
            fontsize=8.5, fontweight='bold' if bold else 'normal', multialignment='center')
    # flecha
    if y != y_positions[-1]:
        next_y = y_positions[y_positions.index(y) + 1] + steps[y_positions.index(y) + 1][1]
        ax.annotate('', xy=(5, y), xytext=(5, y - 0.1),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Etiquetas laterales
ax.text(9.5, 11.3, 'GENÉTICA', ha='left', va='center', fontsize=7.5,
        color='#1976D2', fontweight='bold', rotation=0)
ax.text(9.5, 9.8, 'MOLECULAR', ha='left', va='center', fontsize=7.5,
        color='#E65100', fontweight='bold')
ax.text(9.5, 6.5, 'CELULAR', ha='left', va='center', fontsize=7.5,
        color='#C00000', fontweight='bold')
ax.text(9.5, 3.8, 'TISULAR', ha='left', va='center', fontsize=7.5,
        color='#880E4F', fontweight='bold')
ax.text(9.5, 1.3, 'CLÍNICO', ha='left', va='center', fontsize=7.5,
        color='#B71C1C', fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figura_02c_cascada_bub1b_mva_fenotipo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: results/figura_02c_cascada_bub1b_mva_fenotipo.png')

---
## 5. Conexión fenotipo clínico ↔ mecanismo

| Característica clínica | HPO | Explicación mecanística |
|------------------------|-----|------------------------|
| **Rabdomiosarcoma** | HP:0002859 | CIN en precursores musculares → pérdida RB1/TP53 → proliferación |
| **Prematuridad (32 sem)** | HP:0001622 | CIN en células placentarias → disfunción placentaria → parto prematuro |
| **RCIU severo (~1 kg, SGA)** | HP:0001518 | Aneuploidía en células fetales durante organogénesis → crecimiento reducido |
| **Talla baja** | HP:0004322 | Aneuploidía en condrocitos/células GH → defecto en crecimiento longitudinal |
| **Falla de medro** | HP:0001508 | Aneuploidía multi-tejido → disfunción metabólica sistémica |
| **Nefrocalcinosis** | HP:0000121 | Aneuploidía en células tubulares renales → transporte iónico alterado |
| **Atrofia muscular** | HP:0003202 | CIN en miocitos + efecto del rabdomiosarcoma |
| **Abortos parentales** | HP:0200067 | Progenitores portadores (50% BUBR1) → CIN leve → embriones aneuploides |

**Consistencia total:** El fenotipo encaja perfectamente con la hipótesis BUB1B bialélico.

---
## 6. Resumen para R4 — Base para reposicionamiento farmacológico

Este resumen es la **entrega del Hito H2** (R3 → R4, 12 oct).

In [ ]:
# ── Celda 4: Generar resumen_mecanismo_para_R4.json ────────────────────────

resumen_r4 = {
    'hito': 'H2',
    'fecha': '2026-10-12',
    'para': 'R4 (farmacología y reposicionamiento)',
    'gen_causante': 'BUB1B',
    'proteina': 'BUBR1',
    'tipo_efecto_molecular': 'Pérdida de función (LOF) — haploinsuficiencia compuesta',
    'mecanismo_clave': {
        'via': 'Spindle Assembly Checkpoint (SAC)',
        'complejo_afectado': 'MCC (Mitotic Checkpoint Complex: BUBR1:BUB3:MAD2:CDC20)',
        'efecto_MCC': 'Inestable — inhibición APC/C incompleta',
        'consecuencia_celular': 'Missegregación cromosómica → aneuploidía → mosaicismo'
    },
    'targets_farmacologicos_potenciales': [
        {
            'target': 'APC/C (complejo promotor de anafase)',
            'racional': 'Inhibir APC/C compensaría la inhibición insuficiente del SAC mutado',
            'nota': 'Target muy tóxico — requiere selectividad tumor vs. normal'
        },
        {
            'target': 'CDC20',
            'racional': 'CDC20 libre (no secuestrado por MCC) activa APC/C prematuramente; inhibidores CDC20 simulan SAC activo',
            'nota': 'Apcin, pro-TAME (herramienta experimental)'
        },
        {
            'target': 'PLK1 (Polo-like kinase 1)',
            'racional': 'PLK1 regula la maduración del cinetocoro y la unión microtúbulo. En células con SAC débil, inhibir PLK1 retarda la mitosis dando tiempo a la bi-orientación',
            'nota': 'Volasertib (aprobado Europa para AML); investigado en sarcomas'
        },
        {
            'target': 'Aurora B kinase',
            'racional': 'Aurora B es el "error correction" — detecta uniones incorrectas y las deshace. Potenciar Aurora B podría compensar el SAC débil',
            'nota': 'Barasertib (alisertib en Aurora A); Danusertib'
        },
        {
            'target': 'MPS1 / TTK (SAC kinase)',
            'racional': 'MPS1 inicia la señal del SAC reclutando MAD1/MAD2 al cinetocoro. Activar MPS1 podría fortalecer el checkpoint residual',
            'nota': 'Ojo: inhibidores MPS1 son pro-tumor en células normales; dirección opuesta'
        },
        {
            'target': 'Hsp90 (chaperona)',
            'racional': 'Hsp90 estabiliza BUBR1 y otras kinasas mitóticas. Moduladores Hsp90 podrían estabilizar el alelo missense (p.Asn1002Lys)',
            'nota': 'Geldanamicina, 17-AAG; también candidatos en enfermedades raras con missense'
        },
        {
            'target': 'p53 pathway',
            'racional': 'Las células aneuploides con daño cromosómico activan p53 → apoptosis. Restaurar p53 eliminaría células tumorales con mayor aneuploidía',
            'nota': 'Nutlin-3a/APR-246 (rescate p53 mutante); relevante si el rabdomiosarcoma tiene TP53 wt'
        }
    ],
    'criterios_seleccion_para_R4': [
        'Aprobados FDA/EMA (preferible) o en fase II/III pediátrica',
        'Seguridad en pediatría documentada',
        'Mecanismo que compense SAC débil O que elimine células con alta CIN',
        'Evidencia en rabdomiosarcoma o sarcomas de partes blandas'
    ]
}

out_path = RESULTS_DIR / 'resumen_mecanismo_para_R4.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(resumen_r4, f, indent=2, ensure_ascii=False)

print('=== RESUMEN PARA R4 (Hito H2) ===')
print(f'Guardado en: {out_path}')
print()
print(f'Gen causante: {resumen_r4["gen_causante"]} / Proteína: {resumen_r4["proteina"]}')
print(f'Efecto molecular: {resumen_r4["tipo_efecto_molecular"]}')
print(f'Vía afectada: {resumen_r4["mecanismo_clave"]["via"]}')
print()
print(f'Targets farmacológicos identificados: {len(resumen_r4["targets_farmacologicos_potenciales"])}')
for t in resumen_r4['targets_farmacologicos_potenciales']:
    print(f"  • {t['target']}: {t['nota']}")

---
## 7. Literatura clave

> ⚠ **Citas sin verificar (tarea B1).** Esta tabla se redactó de memoria. Antes de usarla en el informe hay que comprobar cada una en PubMed (autores, revista, volumen, páginas) y que realmente respalde lo que dice la columna *Relevancia*.

| Referencia | Relevancia |
|-----------|------------|
| Hanks et al. (2004) *Nat Genet* 36, 1159–1161 | Primera descripción de BUB1B bialélico en MVA |
| Matsuura et al. (2006) *J Biol Chem* 281, 2457–2466 | Estructura BUBR1 y dominios funcionales |
| Malureanu et al. (2009) *Dev Cell* 16, 787–798 | BUBR1 mínimo funcional para SAC; modelo de knockdown |
| Suijkerbuijk et al. (2012) *Dev Cell* 23, 386–398 | Dominio GLEBS y unión BUB3 — esencial para MCC |
| Elowe et al. (2012) *J Cell Sci* 125, 4507–4517 | Pseudo-kinasa de BUBR1 y su rol en silencing SAC |
| Rivero et al. (2016) *Eur J Hum Genet* 24, 765–772 | Serie clínica MVA con BUB1B; genotipo-fenotipo |
| Holland & Cleveland (2012) *Science* 337, 1433–1434 | CIN como driver oncogénico en aneuploidía |
| Simonetti et al. (2019) *Cancers* 11, 1–17 | Inhibidores kinasas mitóticas en sarcomas pediátricos |

**Para buscar más:** PubMed queries recomendadas:
- `BUB1B OR BUBR1 AND "mosaic variegated aneuploidy"`
- `"spindle assembly checkpoint" AND "drug repurposing"`
- `PLK1 inhibitor AND rhabdomyosarcoma`
- `Aurora kinase inhibitor AND "rare disease" AND pediatric`

In [ ]:
# ── Celda 5: Guardar resumen del Hito H3 ──────────────────────────────────

hito_h3 = {
    'hito': 'H3',
    'fecha': '2026-10-14',
    'titulo': 'Mecanismo biológico completo — BUB1B/BUBR1 y MVA',
    'secciones': [
        'Estructura BUBR1: 5 dominios, 1050 aa, pseudo-kinasa C-terminal',
        'SAC: MCC = BUBR1:BUB3:MAD2:CDC20 inhibe APC/C hasta bi-orientación completa',
        'Variante 1 (p.Leu737Ter): PTC → NMD → alelo nulo (sin proteína)',
        'Variante 2 (p.Asn1002Lys): missense pseudo-kinasa → proteína disfuncional',
        'Mecanismo: haploinsuficiencia compuesta → MCC débil → APC/C prematuro → missegregación → mosaicismo',
        'Fenotipo: rabdomiosarcoma + prematuridad + RCIU + nefrocalcinosis explicados por CIN multi-tejido',
        'Abortos parentales: portadores de 1 alelo → CIN leve → embriones aneuploides'
    ],
    'figuras': [
        'figura_02a_dominios_bubr1.png',
        'figura_02b_sac_normal_vs_bub1b.png',
        'figura_02c_cascada_bub1b_mva_fenotipo.png'
    ],
    'resumen_una_frase': (
        'La heterocigótica compuesta en BUB1B produce haploinsuficiencia casi completa de BUBR1, '
        'desactivando el Spindle Assembly Checkpoint y generando missegregación crónica que causa '
        'el mosaicismo aneuplo (MVA) y la predisposición a rabdomiosarcoma.'
    )
}

out_h3 = RESULTS_DIR / 'hito_H3_mecanismo_completo.json'
with open(out_h3, 'w', encoding='utf-8') as f:
    json.dump(hito_h3, f, indent=2, ensure_ascii=False)

print('=== HITO H3 — MECANISMO COMPLETO ===')
print()
print(hito_h3['resumen_una_frase'])
print()
print('Figuras generadas:')
for fig in hito_h3['figuras']:
    path = RESULTS_DIR / fig
    print(f'  {fig}: {"✅" if path.exists() else "⬜ pendiente"}')
print(f'\nGuardado: {out_h3}')